# Day 6 — CNN Image Classification | RetailMind AI

This notebook builds two TensorFlow/Keras image classifiers:

1. **Fashion-MNIST** — a small 10-class prototype to validate the CNN.
2. **Fashion Product Images Small** — a classifier using real product photographs and `masterCategory` labels.

All model artifacts and evaluation CSV files are saved under `models/vision/`.

> Run the Fashion-MNIST section first. For the real product-image section, download and extract the Kaggle dataset into `data/fashion_product_images/` so that it contains `images/` and `styles.csv`.


## 1. Imports and project paths

In [1]:
import json
import numpy as np
import pandas as pd
import tensorflow as tf

from pathlib import Path
from tensorflow import keras
from tensorflow.keras import layers

# This notebook is expected to live in the repository's notebooks/ directory.
# If it is run from the repository root, use the current directory instead.
CWD = Path.cwd()
ROOT_DIR = CWD.parent if (CWD.parent / "data").exists() else CWD

DATA_DIR = ROOT_DIR / "data"
MODEL_DIR = ROOT_DIR / "models" / "vision"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("TensorFlow:", tf.__version__)
print("Project root:", ROOT_DIR)
print("Model directory:", MODEL_DIR)

TensorFlow: 2.21.0
Project root: c:\Users\KIIT\Desktop\Incremental Code
Model directory: c:\Users\KIIT\Desktop\Incremental Code\models\vision


## 2. Load Fashion-MNIST

In [3]:
(X_train, y_train), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()

print("Training images:", X_train.shape)
print("Training labels:", y_train.shape)
print("Test images:", X_test.shape)
print("Test labels:", y_test.shape)

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 2us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 25s 1us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Training images: (60000, 28, 28)
Training labels: (60000,)
Test images: (10000, 28, 28)
Test labels: (10000,)


## 3. Normalize and prepare Fashion-MNIST images

In [4]:
# Convert pixel values from [0, 255] to [0, 1].
X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

# Add the channel dimension required by Conv2D: (height, width, channels).
X_train = X_train[..., np.newaxis]
X_test = X_test[..., np.newaxis]

CLASS_NAMES = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]
NUM_CLASSES = len(CLASS_NAMES)

print("Prepared training shape:", X_train.shape)
print("Prepared test shape:", X_test.shape)

Prepared training shape: (60000, 28, 28, 1)
Prepared test shape: (10000, 28, 28, 1)


## 4. Build the CNN

In [5]:
def build_cnn(input_shape, num_classes):
    model = keras.Sequential([
        keras.Input(shape=input_shape),

        layers.Conv2D(32, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(64, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),

        layers.Flatten(),
        layers.Dropout(0.5),
        layers.Dense(128, activation="relu"),
        layers.Dense(num_classes, activation="softmax")
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model


def get_early_stopping(patience=5):
    return keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=patience,
        restore_best_weights=True,
        verbose=1
    )


fashion_model = build_cnn(
    input_shape=(28, 28, 1),
    num_classes=NUM_CLASSES
)
fashion_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       204,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 225,034 (879.04 KB)

 Trainable params: 225,034 (879.04 KB)

 Non-trainable params: 0 (0.00 B)

## 5. Train Fashion-MNIST CNN

In [6]:
fashion_history = fashion_model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=30,
    batch_size=64,
    callbacks=[get_early_stopping()],
    verbose=1
)

Epoch 1/30
750/750 ━━━━━━━━━━━━━━━━━━━━ 15s 18ms/step - accuracy: 0.7899 - loss: 0.5760 - val_accuracy: 0.8593 - val_loss: 0.3974
Epoch 2/30
750/750 ━━━━━━━━━━━━━━━━━━━━ 14s 18ms/step - accuracy: 0.8527 - loss: 0.4056 - val_accuracy: 0.8769 - val_loss: 0.3428
Epoch 3/30
750/750 ━━━━━━━━━━━━━━━━━━━━ 12s 16ms/step - accuracy: 0.8692 - loss: 0.3599 - val_accuracy: 0.8866 - val_loss: 0.3123
Epoch 4/30
750/750 ━━━━━━━━━━━━━━━━━━━━ 13s 17ms/step - accuracy: 0.8798 - loss: 0.3236 - val_accuracy: 0.8841 - val_loss: 0.3077
Epoch 5/30
750/750 ━━━━━━━━━━━━━━━━━━━━ 13s 17ms/step - accuracy: 0.8899 - loss: 0.2996 - val_accuracy: 0.9021 - val_loss: 0.2724
Epoch 6/30
750/750 ━━━━━━━━━━━━━━━━━━━━ 12s 16ms/step - accuracy: 0.8961 - loss: 0.2799 - val_accuracy: 0.9022 - val_loss: 0.2648
Epoch 7/30
750/750 ━━━━━━━━━━━━━━━━━━━━ 16s 21ms/step - accuracy: 0.9001 - loss: 0.2681 - val_accuracy: 0.9047 - val_loss: 0.2583
Epoch 8/30
750/750 ━━━━━━━━━━━━━━━━━━━━ 15s 13ms/step - accuracy: 0.9059 - loss: 0.2523 - 

## 6. Evaluate Fashion-MNIST

In [7]:
fashion_test_loss, fashion_test_accuracy = fashion_model.evaluate(
    X_test, y_test, verbose=1
)

print(f"Fashion-MNIST test loss: {fashion_test_loss:.4f}")
print(f"Fashion-MNIST test accuracy: {fashion_test_accuracy:.4f}")

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9154 - loss: 0.2372
Fashion-MNIST test loss: 0.2372
Fashion-MNIST test accuracy: 0.9154


## 7. Fashion-MNIST confusion matrix and CSV outputs

In [8]:
fashion_probabilities = fashion_model.predict(X_test, verbose=0)
fashion_predictions = np.argmax(fashion_probabilities, axis=1)

fashion_confusion = tf.math.confusion_matrix(
    y_test,
    fashion_predictions,
    num_classes=NUM_CLASSES
).numpy()

fashion_confusion_df = pd.DataFrame(
    fashion_confusion,
    index=CLASS_NAMES,
    columns=CLASS_NAMES
)
fashion_confusion_df.to_csv(MODEL_DIR / "fashion_mnist_confusion_matrix.csv")

fashion_class_accuracy = np.diag(fashion_confusion) / np.maximum(
    fashion_confusion.sum(axis=1), 1
)
pd.DataFrame({
    "Category": CLASS_NAMES,
    "Accuracy": fashion_class_accuracy
}).to_csv(MODEL_DIR / "fashion_mnist_class_accuracy.csv", index=False)

fashion_metrics = pd.DataFrame([{
    "Model": "Fashion-MNIST CNN",
    "Test Loss": fashion_test_loss,
    "Test Accuracy": fashion_test_accuracy
}])
fashion_metrics.to_csv(MODEL_DIR / "fashion_mnist_metrics.csv", index=False)

print("Saved Fashion-MNIST evaluation CSV files.")

Saved Fashion-MNIST evaluation CSV files.


## 8. Save Fashion-MNIST model

In [9]:
fashion_model_path = MODEL_DIR / "fashion_mnist_cnn.keras"
fashion_model.save(fashion_model_path)
print("Saved:", fashion_model_path)

Saved: c:\Users\KIIT\Desktop\Incremental Code\models\vision\fashion_mnist_cnn.keras


## 9. Prepare Fashion Product Images dataset

Download [Fashion Product Images Small from Kaggle](https://www.kaggle.com/datasets/paramaggarwal/fashion-product-images-small) and extract it to:

```text
data/
└── fashion_product_images/
    ├── images/
    └── styles.csv
```

This notebook predicts `masterCategory` first. It is a broad product category. You can later change the target to `articleType` for more detailed labels.


In [10]:
PRODUCT_DATA_DIR = DATA_DIR / "fashion_product_images"
IMAGE_DIR = PRODUCT_DATA_DIR / "images"
METADATA_PATH = PRODUCT_DATA_DIR / "styles.csv"

if not IMAGE_DIR.exists() or not METADATA_PATH.exists():
    raise FileNotFoundError(
        "Dataset not found. Expected data/fashion_product_images/images/ "
        "and data/fashion_product_images/styles.csv"
    )

# on_bad_lines='skip' handles malformed metadata rows if present.
metadata = pd.read_csv(METADATA_PATH, on_bad_lines="skip")

required_columns = {"id", "masterCategory"}
missing_columns = required_columns - set(metadata.columns)
if missing_columns:
    raise ValueError(f"Missing required columns in styles.csv: {missing_columns}")

metadata = metadata.dropna(subset=["id", "masterCategory"]).copy()
metadata["id"] = metadata["id"].astype(str).str.replace(r"\.0$", "", regex=True)
metadata["image_path"] = metadata["id"].map(lambda image_id: str(IMAGE_DIR / f"{image_id}.jpg"))

# Keep only rows whose corresponding image file exists.
metadata = metadata[metadata["image_path"].map(lambda path: Path(path).is_file())].copy()
metadata["masterCategory"] = metadata["masterCategory"].astype(str)

print("Usable images:", len(metadata))
print("Categories:")
print(metadata["masterCategory"].value_counts())

Usable images: 44419
Categories:
masterCategory
Apparel           21392
Accessories       11274
Footwear           9219
Personal Care      2403
Free Items          105
Sporting Goods       25
Home                  1
Name: count, dtype: int64


## 10. Encode labels and create train/validation/test splits

In [11]:
# Stable category-to-index mapping. Save this mapping with the trained model.
CLASS_NAMES_PRODUCT = sorted(metadata["masterCategory"].unique().tolist())
class_to_index = {name: index for index, name in enumerate(CLASS_NAMES_PRODUCT)}

metadata["label"] = metadata["masterCategory"].map(class_to_index).astype("int32")

# Shuffle within each class before splitting so smaller categories are represented
# in train, validation, and test where possible.
rng = np.random.default_rng(42)
train_parts, val_parts, test_parts = [], [], []

for _, group in metadata.groupby("label"):
    indices = group.index.to_numpy()
    rng.shuffle(indices)

    n = len(indices)
    train_end = int(n * 0.8)
    val_end = int(n * 0.9)

    train_parts.extend(indices[:train_end])
    val_parts.extend(indices[train_end:val_end])
    test_parts.extend(indices[val_end:])

train_df = metadata.loc[train_parts].sample(frac=1, random_state=42).reset_index(drop=True)
val_df = metadata.loc[val_parts].sample(frac=1, random_state=42).reset_index(drop=True)
test_df = metadata.loc[test_parts].sample(frac=1, random_state=42).reset_index(drop=True)

NUM_PRODUCT_CLASSES = len(CLASS_NAMES_PRODUCT)

print("Categories:", CLASS_NAMES_PRODUCT)
print("Training images:", len(train_df))
print("Validation images:", len(val_df))
print("Test images:", len(test_df))

Categories: ['Accessories', 'Apparel', 'Footwear', 'Free Items', 'Home', 'Personal Care', 'Sporting Goods']
Training images: 35533
Validation images: 4440
Test images: 4446


## 11. Build the `tf.data` image-loading pipeline

In [12]:
IMAGE_SIZE = (128, 128)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE


def load_product_image(image_path, label):
    image_bytes = tf.io.read_file(image_path)
    image = tf.io.decode_jpeg(image_bytes, channels=3)
    image = tf.image.resize(image, IMAGE_SIZE)
    # Keep pixel values in [0, 255]; model Rescaling layer normalizes them.
    return image, label


def make_product_dataset(frame, training=False):
    paths = frame["image_path"].to_numpy()
    labels = frame["label"].to_numpy(dtype=np.int32)

    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))

    if training:
        dataset = dataset.shuffle(
            buffer_size=min(len(frame), 10000),
            seed=42,
            reshuffle_each_iteration=True
        )

    dataset = dataset.map(
        load_product_image,
        num_parallel_calls=AUTOTUNE
    )
    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(AUTOTUNE)
    return dataset


train_ds = make_product_dataset(train_df, training=True)
val_ds = make_product_dataset(val_df)
test_ds = make_product_dataset(test_df)

print("Datasets are ready.")

Datasets are ready.


## 12. Build the product CNN with data augmentation

In [13]:
def build_product_cnn(input_shape, num_classes):
    augmentation = keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.1),
        layers.RandomZoom(0.1),
    ], name="data_augmentation")

    model = keras.Sequential([
        keras.Input(shape=input_shape),

        augmentation,
        layers.Rescaling(1.0 / 255),

        layers.Conv2D(32, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(64, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(128, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),

        layers.Flatten(),
        layers.Dropout(0.5),
        layers.Dense(128, activation="relu"),
        layers.Dense(num_classes, activation="softmax")
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model


product_model = build_product_cnn(
    input_shape=(128, 128, 3),
    num_classes=NUM_PRODUCT_CLASSES
)
product_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ data_augmentation (Sequential)  │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │     3,211,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,305,543 (12.61 MB)

 Trainable params: 3,305,543 (12.61 MB)

 Non-trainable params: 0 (0.00 B)

## 13. Train the product CNN

In [ ]:
product_history = product_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=[get_early_stopping(patience=5)],
    verbose=1
)

Epoch 1/20


c:\Users\KIIT\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


1111/1111 ━━━━━━━━━━━━━━━━━━━━ 387s 347ms/step - accuracy: 0.8496 - loss: 0.4171 - val_accuracy: 0.9243 - val_loss: 0.2482
Epoch 2/20
1111/1111 ━━━━━━━━━━━━━━━━━━━━ 332s 299ms/step - accuracy: 0.9252 - loss: 0.2338 - val_accuracy: 0.9462 - val_loss: 0.1795
Epoch 3/20
1111/1111 ━━━━━━━━━━━━━━━━━━━━ 275s 248ms/step - accuracy: 0.9410 - loss: 0.1887 - val_accuracy: 0.9536 - val_loss: 0.1611
Epoch 4/20
1111/1111 ━━━━━━━━━━━━━━━━━━━━ 273s 246ms/step - accuracy: 0.9509 - loss: 0.1582 - val_accuracy: 0.9610 - val_loss: 0.1324
Epoch 5/20
1111/1111 ━━━━━━━━━━━━━━━━━━━━ 308s 277ms/step - accuracy: 0.9576 - loss: 0.1391 - val_accuracy: 0.9660 - val_loss: 0.1155
Epoch 6/20
1111/1111 ━━━━━━━━━━━━━━━━━━━━ 20527s 18s/step - accuracy: 0.9622 - loss: 0.1247 - val_accuracy: 0.9669 - val_loss: 0.1177
Epoch 7/20
1111/1111 ━━━━━━━━━━━━━━━━━━━━ 59020s 53s/step - accuracy: 0.9650 - loss: 0.1147 - val_accuracy: 0.9732 - val_loss: 0.0995
Epoch 8/20
 440/1111 ━━━━━━━━━━━━━━━━━━━━ 3:23 303ms/step - accuracy: 0.9

## 14. Evaluate product CNN

In [ ]:
product_test_loss, product_test_accuracy = product_model.evaluate(
    test_ds,
    verbose=1
)

print(f"Product CNN test loss: {product_test_loss:.4f}")
print(f"Product CNN test accuracy: {product_test_accuracy:.4f}")

## 15. Product confusion matrix and evaluation CSV files

In [ ]:
# test_ds is not shuffled, so predictions stay in the same order as test_df.
product_probabilities = product_model.predict(test_ds, verbose=1)
product_predictions = np.argmax(product_probabilities, axis=1)
product_true_labels = test_df["label"].to_numpy(dtype=np.int32)

product_confusion = tf.math.confusion_matrix(
    product_true_labels,
    product_predictions,
    num_classes=NUM_PRODUCT_CLASSES
).numpy()

product_confusion_df = pd.DataFrame(
    product_confusion,
    index=CLASS_NAMES_PRODUCT,
    columns=CLASS_NAMES_PRODUCT
)
product_confusion_df.to_csv(
    MODEL_DIR / "fashion_product_confusion_matrix.csv"
)

product_class_accuracy = np.diag(product_confusion) / np.maximum(
    product_confusion.sum(axis=1), 1
)
pd.DataFrame({
    "Category": CLASS_NAMES_PRODUCT,
    "Accuracy": product_class_accuracy
}).to_csv(
    MODEL_DIR / "fashion_product_class_accuracy.csv",
    index=False
)

product_metrics = pd.DataFrame([{
    "Model": "Fashion Product CNN",
    "Test Loss": product_test_loss,
    "Test Accuracy": product_test_accuracy
}])
product_metrics.to_csv(
    MODEL_DIR / "fashion_product_metrics.csv",
    index=False
)

print("Saved product evaluation CSV files.")

## 16. Save product model and class mapping

In [ ]:
product_model_path = MODEL_DIR / "fashion_product_cnn.keras"
product_model.save(product_model_path)

class_mapping_path = MODEL_DIR / "fashion_product_classes.json"
with open(class_mapping_path, "w", encoding="utf-8") as file:
    json.dump(CLASS_NAMES_PRODUCT, file, indent=4)

print("Saved model:", product_model_path)
print("Saved class mapping:", class_mapping_path)

## 17. Predict the category of one product image

In [ ]:
def predict_product(image_path, model=product_model, class_names=CLASS_NAMES_PRODUCT):
    image_path = Path(image_path)

    image_bytes = tf.io.read_file(str(image_path))
    image = tf.io.decode_jpeg(image_bytes, channels=3)
    image = tf.image.resize(image, IMAGE_SIZE)

    # Model includes its own Rescaling layer.
    image = tf.expand_dims(image, axis=0)

    probabilities = model.predict(image, verbose=0)[0]
    predicted_index = int(np.argmax(probabilities))

    category = class_names[predicted_index]
    confidence = float(probabilities[predicted_index])

    print("Predicted category:", category)
    print(f"Confidence: {confidence:.2%}")

    return category, confidence


sample_image_path = test_df.iloc[0]["image_path"]
predict_product(sample_image_path)

## 18. Day 6 results summary

In [ ]:
results = pd.DataFrame([
    {
        "Model": "Fashion-MNIST CNN",
        "Test Loss": fashion_test_loss,
        "Test Accuracy": fashion_test_accuracy
    },
    {
        "Model": "Fashion Product CNN",
        "Test Loss": product_test_loss,
        "Test Accuracy": product_test_accuracy
    }
])

display(results)
results.to_csv(MODEL_DIR / "day6_results.csv", index=False)

print("Day 6 artifacts saved in:", MODEL_DIR)

## 19. README update

After running the notebook, copy the **actual** test accuracies from the results table into your main `README.md`.

Suggested section:

```markdown
## Day 6 — CNN Image Classification

- Dataset 1: Fashion-MNIST
- Dataset 2: Fashion Product Images Small
- Framework: TensorFlow / Keras
- Architecture: Conv2D → MaxPooling2D → Dropout → Dense → Softmax
- Augmentation: RandomFlip, RandomRotation, RandomZoom

| Model | Test Accuracy |
|---|---:|
| Fashion-MNIST CNN | ADD_ACTUAL_RESULT |
| Fashion Product CNN | ADD_ACTUAL_RESULT |

Artifacts are stored in `models/vision/`.
```

### Notes
- Run cells from top to bottom.
- The Fashion Product Images dataset must be downloaded separately from Kaggle.
- The real-image model can take significantly longer to train than Fashion-MNIST.
- If memory is limited, reduce `IMAGE_SIZE` to `(96, 96)` or `BATCH_SIZE` to `16`.
- The product model predicts `masterCategory`; it does not yet perform fine-grained `articleType` classification.
